# 03 · Run the fine-tuned Qwen on Layout B (`neighborhood_express`)

**Run:** `Runtime → Change runtime type → A100 GPU` (an L4 also works), then `Runtime → Run all`.
When asked, upload the `simply_qwen25_7b_impulse_v2_adapter_*.zip` that notebook 01 downloaded.
**Output:** your browser downloads `simply_impulse_v2_layout_B_<time>.zip`, and the last cell prints a summary
to paste into the Simply chat.

**What it runs:** the same 10 frozen benchmark shoppers used by every earlier Layout A/B study (seed `20261030`),
120 actions each, one model decision per action, `guided_nav_v2` assistance, `unique_single_checkout_v1`.
The model only **proposes**. The Python core applies every action and writes the canonical run record.
Impulse buys are counted by Python (`off_list_units`). None of these shoppers were in the training data.

**Time:** about 45–75 min. No paid API calls. If the run stops early, the partial record is still packaged.

In [ ]:
import subprocess, time
SMI = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU:", SMI or "none")
assert SMI, "No GPU. Runtime → Change runtime type → A100 GPU."
assert int(SMI.split(",")[1].split()[0]) >= 22000, "Use an A100 or L4 GPU runtime."
STAMP = time.strftime("%Y%m%d_%H%M%S")

In [ ]:
LAYOUT = "neighborhood_express"
LAYOUT_LETTER = "B"
SHOPPERS, TURN_BUDGET, RUN_SEED = 10, 120, 20261030   # study seed 20261029 + repetition 1, as in earlier studies
NAV_POLICY = "guided_nav_v2"

### 1 · Install

In [ ]:
import subprocess, sys
PINNED = ["unsloth==2026.11.1", "unsloth_zoo==2026.11.1", "transformers==5.17.0", "peft==0.21.2",
          "accelerate==1.15.0", "bitsandbytes==0.50.2", "datasets==4.8.5", "sentencepiece", "protobuf"]
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINNED])
    print("Installed the pinned stack (same versions as the Nov 4 pilot).")
except subprocess.CalledProcessError:
    print("WARNING: pinned install failed; using current releases. Versions are recorded in the output ZIP.")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "unsloth", "datasets",
                           "peft", "accelerate", "bitsandbytes", "sentencepiece", "protobuf"])

### 2 · Unpack Simply core

In [ ]:
#@title Unpack the bundled Simply core (hash-verified; do not edit)
import base64, hashlib, io, shutil, sys, zipfile
from pathlib import Path
BUNDLE_SHA256 = "5f29cff8f905b33d6dd8cd55185883d30d8fb711ade9d89072cab041cc6c30e0"
SOURCE_SHA256 = "b72a2ba4ce6a55ea1075461d63c4c6c765eaa4beaedce79621721dbc289e9960"   # frozen prompt + guided_nav_v2 code; must match across all three notebooks
BUNDLE_B64 = "..."
_blob = base64.b64decode(BUNDLE_B64)
assert hashlib.sha256(_blob).hexdigest() == BUNDLE_SHA256, "Embedded bundle is corrupted"
PROJECT = Path("/content/simply")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
PROJECT.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(_blob)) as _z:
    _z.extractall(PROJECT)
del _blob, BUNDLE_B64
(PROJECT / "simulation" / "runs").mkdir(exist_ok=True)
sys.path.insert(0, str(PROJECT / "simulation"))
import evaluation.impulse_eval as ie
print("Simply core unpacked to", PROJECT)

### 3 · Upload and verify the adapter from notebook 01

In [ ]:
import json, zipfile
from google.colab import files
print("Upload the adapter ZIP from notebook 01 (simply_qwen25_7b_impulse_v2_adapter_*.zip)")
uploaded = files.upload()
archives = [(n, b) for n, b in uploaded.items() if n.endswith(".zip")]
assert len(archives) == 1, "Upload exactly one adapter ZIP"
ADAPTER_ROOT = Path(f"/content/adapter_{STAMP}")
with zipfile.ZipFile(io.BytesIO(archives[0][1])) as archive:
    for name in archive.namelist():
        assert not name.startswith("/") and ".." not in Path(name).parts, f"unsafe path in ZIP: {name}"
    archive.extractall(ADAPTER_ROOT)
del uploaded, archives
ADAPTER_DIR = ADAPTER_ROOT / "adapter"
experiment = json.loads((ADAPTER_ROOT / "experiment.json").read_text())
assert experiment["training_mode"] == "impulse_v2" and experiment["dataset"] == "simply_impulse_teacher_v2", "not an impulse-v2 adapter"
assert experiment["frozen_source_sha256"] == SOURCE_SHA256, "adapter was trained against different Simply code"
ADAPTER_SHA256 = hashlib.sha256((ADAPTER_DIR / "adapter_model.safetensors").read_bytes()).hexdigest()
assert ADAPTER_SHA256 == experiment["adapter_sha256"], "adapter file differs from the one notebook 01 recorded"
print("Adapter OK:", ADAPTER_SHA256[:16], "| base revision", experiment["base_revision"][:12])

### 4 · Load base + adapter (exact tensor check)

In [ ]:
from unsloth import FastLanguageModel
import torch, importlib.metadata, os, threading, time
from peft import PeftModel
from safetensors.torch import load_file
from transformers import AutoTokenizer

gpu = torch.cuda.get_device_properties(0)
model, _ = FastLanguageModel.from_pretrained(model_name=experiment["base_model"], revision=experiment["base_revision"],
                                             max_seq_length=16384, dtype=None, load_in_4bit=True)
model = PeftModel.from_pretrained(model, str(ADAPTER_DIR), is_trainable=False)
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_DIR))
assert hashlib.sha256(tokenizer.chat_template.encode()).hexdigest() == experiment["chat_template_sha256"], "chat template differs from training"
saved, live = load_file(str(ADAPTER_DIR / "adapter_model.safetensors")), dict(model.named_parameters())
mismatch = sum(1 for name, tensor in saved.items()
               if (p := live.get(name.replace(".lora_A.", ".lora_A.default.").replace(".lora_B.", ".lora_B.default."))) is None
               or not torch.equal(p.detach().cpu().to(tensor.dtype), tensor))
assert saved and mismatch == 0, f"{mismatch} LoRA tensors did not load exactly"
FastLanguageModel.for_inference(model); model.eval()
print(f"Loaded base + {len(saved)} LoRA tensors exactly on {gpu.name}")

### 5 · Serve the model on loopback

In [ ]:
from http.server import ThreadingHTTPServer
from agents.cuda_runtime import CUDARuntime, IMPULSE_MODEL, SETTINGS, BASE_MODEL
from mps_server import handler_for
versions = {p: importlib.metadata.version(p) for p in ("unsloth", "transformers", "peft", "bitsandbytes", "torch")}
identity = {"model": IMPULSE_MODEL, "device": "cuda", "base_model": BASE_MODEL, "base_revision": experiment["base_revision"],
    "training_base_model": experiment["base_model"], "training_mode": "impulse_v2", "dataset": experiment["dataset"],
    "adapter_sha256": ADAPTER_SHA256, "chat_template_sha256": experiment["chat_template_sha256"],
    "frozen_source_sha256": SOURCE_SHA256, "generation": SETTINGS, "gpu": gpu.name, "packages": versions,
    "deployment_note": "impulse_v2 QLoRA adapter on the NF4 training base, served on Colab CUDA."}
runtime = CUDARuntime(model, tokenizer, identity)
server = ThreadingHTTPServer(("127.0.0.1", 8768), handler_for(runtime, IMPULSE_MODEL, "CUDA"))
server.daemon_threads = True
threading.Thread(target=server.serve_forever, daemon=True).start()
os.environ["SIMPLY_CUDA_URL"] = "http://127.0.0.1:8768"
from agents.cuda_provider import impulse_health
print("Serving:", impulse_health()["model"])

### 6 · Run 10 shoppers on Layout B through the Python core

In [ ]:
import json, threading, time, traceback
from core.config import DECISION_RUNS_DIR
from core.grid import build_store
from core.session import SimulationSession
from core.shopping import SIMPLE_SHOPPING
from evaluation.benchmark import load_benchmark, instantiate_benchmark

OUT = Path(f"/content/simply_impulse_v2_layout_{LAYOUT_LETTER}_{STAMP}")
OUT.mkdir(parents=True)
RUN_PATH = OUT / f"run_layout_{LAYOUT_LETTER}.jsonl"
benchmark = load_benchmark(PROJECT / "simulation/benchmarks/two_layouts_15_v2.json")
grid = build_store(LAYOUT)
shoppers = instantiate_benchmark(benchmark, SHOPPERS, grid, TURN_BUDGET, SIMPLE_SHOPPING)
print("Shoppers:", [(s.customer.customer_id, s.customer.shopping_style) for s in shoppers])
session = SimulationSession(LAYOUT, SHOPPERS, TURN_BUDGET, TURN_BUDGET, 1, grid=grid, shoppers=shoppers, decide=None,
    provider_name="local_mps", model_name=IMPULSE_MODEL, seed=RUN_SEED, navigation_policy=NAV_POLICY,
    allow_action_plans=False, shopping_rules=SIMPLE_SHOPPING, output_path=RUN_PATH,
    ledger_path=DECISION_RUNS_DIR / "gemini_campaign.sqlite3", requests_per_minute=0,
    experiment={"kind": "qwen_impulse_v2_layout_run", "layout": LAYOUT, "benchmark_sha256": benchmark["sha256"],
                "cohort_size": SHOPPERS, "seed": RUN_SEED, "adapter_sha256": ADAPTER_SHA256})

def watch():
    while True:
        time.sleep(120)
        try:
            print(time.strftime("%H:%M:%S"), "decisions recorded:", sum(1 for _ in RUN_PATH.open()) - 1, flush=True)
        except Exception:
            pass
threading.Thread(target=watch, daemon=True).start()

RUN_ERROR, started = None, time.time()
try:
    session.run()
except BaseException as error:          # keep the partial record and still package it
    RUN_ERROR = f"{type(error).__name__}: {error}"
    traceback.print_exc()
print(f"Finished in {(time.time() - started) / 60:.1f} min | status {session.status} | error {RUN_ERROR}")

### 7 · Impulse report, package and download

In [ ]:
import hashlib, json, platform, zipfile
report = ie.trip_report(RUN_PATH)
(OUT / "trip_report.json").write_text(json.dumps(report, indent=2))
(OUT / "served_identity.json").write_text(json.dumps(identity, indent=2))
(OUT / "adapter_experiment.json").write_text(json.dumps(experiment, indent=2))
(OUT / "environment.json").write_text(json.dumps({"python": platform.python_version(), "gpu": gpu.name,
    "packages": versions, "run_error": RUN_ERROR, "status": session.status}, indent=2))
members = sorted(p for p in OUT.iterdir() if p.is_file())
(OUT / "MANIFEST.json").write_text(json.dumps({p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in members}, indent=2))
ZIP_PATH = Path(f"/content/{OUT.name}.zip")
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as archive:
    for p in members + [OUT / "MANIFEST.json"]:
        archive.write(p, f"{OUT.name}/{p.name}")
print("Created", ZIP_PATH, f"({ZIP_PATH.stat().st_size / 2**20:.1f} MiB)")
summary = {"layout": LAYOUT, "run_id": report["run_id"], "status": report["status"], "model": report["model"],
           "adapter_sha256": ADAPTER_SHA256[:16], "overall": report["overall"], "by_style": report["by_style"],
           "failures": report["failures"], "run_error": RUN_ERROR}
print("\n=== PASTE THIS INTO THE SIMPLY CHAT ===\n" + json.dumps(summary, indent=1))
files.download(str(ZIP_PATH))